# DBS Agent: SFT → GRPO → HF Hub (Colab)

End‑to‑end Colab notebook for the **Parkinson's Motor** OpenEnv environment.

Pipeline:
1. Clone the env from Hugging Face Spaces and install deps
2. Configure paths / model / hyperparameters
3. Build env helpers (in‑process, no HTTP server) that mirror `inference.py`
4. Quick heuristic baseline (also our SFT teacher)
5. Generate a rejection‑sampled SFT dataset
6. SFT **Qwen2.5‑3B‑Instruct** in 4‑bit with **Unsloth + LoRA**
7. **GRPO** with per‑step return‑to‑go advantages, KL anchor to the SFT model, format‑compliance reward, and an `easy → medium → hard` curriculum
8. **Validation & evaluation suite** — base Qwen vs SFT vs SFT+GRPO vs `safety_aware` vs `no_dbs` with reward curves, component breakdown, success rates, and before/after trajectory plots (everything saved to disk for the README)
9. Merge LoRA + push to the Hugging Face Hub (model card auto‑includes the eval table and plots)

This notebook satisfies the hackathon judging checklist directly: training script that connects to the OpenEnv environment, observable training progress (loss + grader curves), a trained‑vs‑baseline comparison on multiple axes, and saved labeled plots ready to embed in the README.

**Runtime:** runs on Colab and Kaggle (auto-detects). T4 (16 GB) is enough; L4 / A100 ≈2–3× faster. With the FAST defaults the whole notebook takes ≈ 25–45 min on a T4.

**Kaggle setup:** Settings → Accelerator → **GPU T4 x2** (or P100). Settings → Internet → **On**. Add-ons → Secrets → add `HF_TOKEN` (write-scope). Then run all.

## 1 · GPU check + clone the OpenEnv repo

In [ ]:
import os, subprocess, sys, shutil, pathlib

# ---- Detect runtime: Colab vs Kaggle vs local ----------------------------
ON_COLAB  = 'COLAB_GPU' in os.environ or os.path.isdir('/content')
ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.isdir('/kaggle')
# (Colab only) Optional: persist EVERYTHING to your Google Drive so a
# disconnect doesn't wipe SFT data, adapters, plots, or logs.
MOUNT_DRIVE = False  # flip to True to keep run artifacts on /content/drive
if ON_KAGGLE:
    BASE_DIR = pathlib.Path('/kaggle/working')
elif ON_COLAB:
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        BASE_DIR = pathlib.Path('/content/drive/MyDrive/dbs_runs')
    else:
        BASE_DIR = pathlib.Path('/content')
else:
    BASE_DIR = pathlib.Path.cwd()
BASE_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME = 'kaggle' if ON_KAGGLE else ('colab' if ON_COLAB else 'local')
print(f'Runtime: {RUNTIME}   BASE_DIR: {BASE_DIR}')

print('=== GPU info ===')
try:
    print(subprocess.check_output(['nvidia-smi']).decode().split('\n')[0:12])
except Exception as e:
    print('nvidia-smi failed (no GPU?):', e)
    if ON_KAGGLE:
        print('  -> On Kaggle: Settings -> Accelerator -> GPU T4 x2 (or P100). Also turn Internet ON.')

REPO_URL  = 'https://huggingface.co/spaces/virustechhacks/parkinsons_Motor'
CLONE_DIR = str(BASE_DIR / 'parkinsons_motor_space')

if os.path.isdir(CLONE_DIR):
    print(f'Already cloned: {CLONE_DIR}')
else:
    print(f'Cloning {REPO_URL} -> {CLONE_DIR} ...')
    rc = subprocess.call(['git', 'clone', '--depth', '1', REPO_URL, CLONE_DIR])
    assert rc == 0, 'git clone failed (on Kaggle: turn Internet ON in notebook settings)'

if CLONE_DIR not in sys.path:
    sys.path.insert(0, CLONE_DIR)
print('Repo contents:', sorted(os.listdir(CLONE_DIR))[:20])

## 2 · Install dependencies

Unsloth gives us 2–3× faster LoRA + 4‑bit weight loading on Colab and Kaggle. We also install TRL for tokenizer/SFT helpers, openenv‑core for the env interface, and HF Hub for pushing weights. The `unsloth[colab-new]` extra is the recommended one for both Colab and Kaggle environments (despite the name).

In [ ]:
%%capture
# Install Unsloth (pulls compatible torch, transformers, trl, peft, bitsandbytes, accelerate)
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# Force-pin a few co-deps so versions stay aligned with Unsloth
!pip install -q -U --no-deps "trl>=0.11,<0.14" "peft>=0.12.0" "accelerate>=0.32.0"
# Env runtime + scientific deps
!pip install -q -U "openenv-core[core]>=0.2.2" numpy scipy huggingface_hub matplotlib pandas datasets
print('Installed.')

In [ ]:
import torch, transformers, trl, peft
print('torch       :', torch.__version__, 'cuda?', torch.cuda.is_available())
print('transformers:', transformers.__version__)
print('trl         :', trl.__version__)
print('peft        :', peft.__version__)
if torch.cuda.is_available():
    print('GPU         :', torch.cuda.get_device_name(0))

## 3 · Hugging Face login (for weight push at the end)

In [ ]:
import os
from huggingface_hub import login, whoami

# Try secrets in order: Colab userdata -> Kaggle secrets -> env var -> interactive paste.
hf_token = None
try:
    from google.colab import userdata          # Colab
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    pass
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient   # Kaggle
        hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    import getpass
    hf_token = getpass.getpass('Enter your Hugging Face token (write scope, hidden): ').strip()

assert hf_token, ('A Hugging Face token is required.\n'
                  '  Colab : Tools -> Secrets -> add HF_TOKEN\n'
                  '  Kaggle: Add-ons -> Secrets -> add HF_TOKEN (label exactly HF_TOKEN)\n'
                  '  HF UI : Settings -> Access Tokens -> create one with WRITE scope.')
login(token=hf_token, add_to_git_credential=True)
os.environ['HF_TOKEN'] = hf_token
print('Logged in as:', whoami()['name'])

## 4 · Configuration

All knobs live here. The current FAST defaults target a single T4 session of ≈25–45 min on either Colab or Kaggle. Scale up `sft_epochs`, `grpo_curriculum`, and `eval_seeds` for stronger results.

In [ ]:
from types import SimpleNamespace
import pathlib, os

# BASE_DIR was set in cell 1 (Colab=/content, Kaggle=/kaggle/working, local=cwd).
WORK_DIR = BASE_DIR / 'dbs_run'; WORK_DIR.mkdir(parents=True, exist_ok=True)

CFG = SimpleNamespace(
    # ---- model ---------------------------------------------------------
    base_model        = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit',  # 4-bit Qwen-3B
    max_seq_len       = 1024,     # FAST: 1024 (was 2048). Bump to 2048 if prompts overflow.
    lora_r            = 8,        # FAST: r=8 (was 16). Halves LoRA params -> ~30% faster fwd/bwd.
    lora_alpha        = 16,       # keep alpha = 2 * r
    # ---- env / tasks ---------------------------------------------------
    public_tasks      = ['easy', 'medium', 'hard'],
    eval_seeds        = [0],      # FAST mode: 1 seed. Bump to [0,1,2,3] for error bars.
    # ---- SFT data generation ------------------------------------------
    # FAST mode: ~5x fewer teacher episodes than the 'full' run (60/60/30).
    sft_episodes_per_task = {'easy': 12, 'medium': 12, 'hard': 6},
    sft_min_grader_for_full_keep = 0.55,   # keep all steps from episodes scoring >= this
    sft_min_grader_for_any_keep  = 0.30,   # below this: drop the episode entirely
    sft_max_examples              = 1500,  # FAST mode cap (was 6000); raise for stronger SFT
    # ---- SFT training -------------------------------------------------
    sft_epochs        = 1,        # FAST mode: 1 epoch (was 2). Bump to 2-3 for stronger SFT.
    sft_batch_size    = 4,
    sft_grad_accum    = 4,
    sft_lr            = 2e-4,
    sft_warmup_ratio  = 0.05,
    # ---- GRPO ---------------------------------------------------------
    # FAST mode: 4 updates total on easy+medium with batched gen + batched fwd/bwd.
    # ~6-10 min on T4. (`hard` is dropped from training but still EVALUATED.)
    # Reference 'strong' config (uncomment to use):
    #   ('easy', 10, 4, 0.9), ('medium', 6, 3, 0.9), ('hard', 3, 2, 0.85)
    grpo_curriculum   = [
        # task_id, n_updates, group_size, sample_temp
        ('easy',   2, 2, 0.9),
        ('medium', 2, 2, 0.9),
    ],
    grpo_lr           = 5e-6,
    grpo_max_new_tok  = 32,            # FAST: 32 (was 48); JSON action fits comfortably in 32 tokens
    grpo_kl_coef      = 0.04,
    grpo_format_bonus = 0.05,
    grpo_format_penalty = 0.10,
    grpo_gamma_rtg    = 0.97,
    grpo_grad_clip    = 1.0,
    grpo_promote_score = 0.70,         # advance curriculum early if rolling grader >= this
    # ---- GRPO speed knobs (batching) ----------------------------------
    grpo_pol_micro_bsz = 2,            # forward+backward batch (with grad). Lower if OOM.
    grpo_ref_micro_bsz = 4,            # forward batch for reference logp (no grad).
    grpo_gen_batch_size = 0,           # 0 = use group_size; set lower if generation OOMs.
    # ---- HF push ------------------------------------------------------
    hf_repo_id        = 'YOUR_HF_USERNAME/dbs-qwen3b-sft-grpo',  # << EDIT before running cell 14
    push_merged_16bit = True,
    # ---- paths --------------------------------------------------------
    sft_data_path     = str(WORK_DIR / 'sft_dataset.jsonl'),
    sft_adapter_dir   = str(WORK_DIR / 'sft_adapter'),
    grpo_adapter_dir  = str(WORK_DIR / 'grpo_adapter'),
    merged_dir        = str(WORK_DIR / 'merged_16bit'),
    plots_dir         = str(WORK_DIR / 'plots'),
    eval_json         = str(WORK_DIR / 'final_eval.json'),
)
pathlib.Path(CFG.plots_dir).mkdir(exist_ok=True, parents=True)
print('CFG ready. WORK_DIR =', WORK_DIR)
print('Edit CFG.hf_repo_id below before pushing to the Hub.')

## 4b · Run directory, logs, and checkpoint layout

Everything from this run lands under `WORK_DIR/run_<timestamp>/` so multiple
Colab runs don't clobber each other. The layout the rest of the notebook writes to:

```
run_<UTC>/
  logs/
    sft/sft_steps.jsonl              # one row per SFT optimizer step
    sft/sft_summary.json             # final summary + total wall time
    grpo/rollouts.jsonl              # one row per rollout episode
    grpo/updates.jsonl               # one row per GRPO optimizer step
    grpo/steps_sample.jsonl          # sampled per-step transitions (debug)
    eval/episode_rows.jsonl          # one row per eval episode (all 6 policies)
    eval/inference_<pol>_<task>.jsonl  # full LLM-trace replay for one seed
  checkpoints/
    sft_final/                       # SFT LoRA adapter
    grpo_after_easy/                 # adapter checkpoint after each curriculum stage
    grpo_after_medium/
    grpo_after_hard/
    grpo_final/                      # final GRPO adapter
    merged_16bit/                    # merged weights for HF push
  plots/                             # all PNGs + summary CSVs
  cfg.json                           # exact config used for the run
```

In [ ]:
import json, time, pathlib, os, datetime, dataclasses
from types import SimpleNamespace

# Per-run root, timestamped so re-runs don't overwrite each other
RUN_ID  = datetime.datetime.utcnow().strftime('%Y%m%d-%H%M%SUTC')
RUN_DIR = pathlib.Path(WORK_DIR) / f'run_{RUN_ID}'
(RUN_DIR / 'logs/sft').mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs/grpo').mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs/eval').mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'plots').mkdir(parents=True, exist_ok=True)

# Re-bind the CFG paths so EVERYTHING from here on writes to RUN_DIR
CFG.run_id           = RUN_ID
CFG.run_dir          = str(RUN_DIR)
CFG.sft_adapter_dir  = str(RUN_DIR / 'checkpoints/sft_final')
CFG.grpo_adapter_dir = str(RUN_DIR / 'checkpoints/grpo_final')
CFG.merged_dir       = str(RUN_DIR / 'checkpoints/merged_16bit')
CFG.plots_dir        = str(RUN_DIR / 'plots')
CFG.eval_json        = str(RUN_DIR / 'logs/eval/final_eval_summary.json')

# Persist exact config for reproducibility
with open(RUN_DIR / 'cfg.json', 'w') as f:
    cfg_dict = {k: v for k, v in vars(CFG).items() if not k.startswith('_')}
    json.dump(cfg_dict, f, indent=2, default=str)


class JsonlLogger:
    """Append-only JSONL logger. Flushes after every write so Colab disconnects don't lose data."""
    def __init__(self, path):
        self.path = pathlib.Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self._fh = open(self.path, 'a', encoding='utf-8')
        self.n = 0
    def log(self, **row):
        row.setdefault('ts', time.time())
        self._fh.write(json.dumps(row, default=float) + '\n')
        self._fh.flush()
        self.n += 1
    def close(self):
        try: self._fh.close()
        except Exception: pass

# Logger handles the rest of the notebook will use
LOG = SimpleNamespace(
    sft_steps     = JsonlLogger(RUN_DIR / 'logs/sft/sft_steps.jsonl'),
    grpo_rollouts = JsonlLogger(RUN_DIR / 'logs/grpo/rollouts.jsonl'),
    grpo_updates  = JsonlLogger(RUN_DIR / 'logs/grpo/updates.jsonl'),
    grpo_steps    = JsonlLogger(RUN_DIR / 'logs/grpo/steps_sample.jsonl'),
    eval_rows     = JsonlLogger(RUN_DIR / 'logs/eval/episode_rows.jsonl'),
)

print('RUN_DIR     :', RUN_DIR)
print('config dump :', RUN_DIR / 'cfg.json')
print('SFT adapter :', CFG.sft_adapter_dir)
print('GRPO adapter:', CFG.grpo_adapter_dir)
print('logs        :', RUN_DIR / 'logs')


## 5 · Environment helpers

We use the **in‑process** `ParkinsonsMotorEnvironment` (no FastAPI/uvicorn subprocess required — simpler and faster on Colab).

Prompt format **mirrors `parkinsons_Motor/inference.py` exactly** so train‑time and eval‑time prompts are identical.

In [ ]:
import textwrap, json, math
from typing import Optional

from parkinsons_Motor.server.parkinsons_Motor_environment import ParkinsonsMotorEnvironment
from parkinsons_Motor.core.models import ParkinsonsMotorAction, ParkinsonsMotorObservation
from parkinsons_Motor.tasks import get_task

SYSTEM_PROMPT = textwrap.dedent("""
    You are an expert closed-loop DBS controller managing Parkinsonian motor symptoms in real time.
    Every step is a short clinical control decision: suppress pathological activity, preserve movement,
    avoid overstimulation, and keep enough safety budget for the rest of the episode.

    Return JSON only:
    {\"dbs_amplitude\": X, \"dbs_pulse_width\": X, \"dbs_frequency\": X}

    Important:
    - Do not output explanations.
    - Do not include motor_command.
    - Prefer pulse_width = 0.13 and frequency = 130.
    - Keep amplitude changes smooth unless there is a clear rescue or safety reason.

    Clinical meaning:
    - beta_arv and tremor_arv should go down.
    - force_preserved should stay high.
    - side_effect_load and gamma_arv warn about overstimulation.
    - positive beta/tremor trends mean worsening.
    - positive side_effect_rate means the burden is still rising.

    Control priorities:
    1. Prevent unsafe overstimulation.
    2. Do not leave symptoms undertreated when clearly elevated.
    3. Restore useful motor function.
    4. Taper toward the lowest effective dose once stable.

    Policy:
    - If gamma_arv is high or side effects are near budget, reduce.
    - If tremor_arv > 0.55 or beta_arv > 0.60, usually treat actively with at least about 1.2 mA unless unsafe.
    - If symptoms worsen and safety is acceptable, increase gradually.
    - If symptoms improve and side effects continue to rise, hold or reduce.
    - If control is stable, taper slowly instead of staying high forever.
""").strip()

_TASK_CONTEXT = {
    'easy':   'EASY / Calm Start. Responsive patient early in symptom build-up. Goal: stabilize mild pathology quickly without wasting safety budget. Ceiling: 1.5 mA. Side-effect budget: 0.55.',
    'medium': 'MEDIUM / Rescue Phase. Symptoms are worsening and force is at risk. Goal: rescue actively, then taper once recovery begins. Ceiling: 1.8 mA. Side-effect budget: 0.60.',
    'hard':   'HARD / Full Episode. Long-horizon management across onset, escalation, peak symptoms, and recovery. Goal: keep the patient functional through the whole episode, not just a short rescue. Ceiling: 2.4 mA. Side-effect budget: 0.55.',
}

def obs_to_dict(obs: ParkinsonsMotorObservation) -> dict:
    return {
        'beta_arv': float(obs.beta_arv),
        'tremor_arv': float(obs.tremor_arv),
        'force_preserved': float(obs.force_preserved),
        'side_effect_load': float(obs.side_effect_load),
        'beta_trend': float(obs.beta_trend),
        'tremor_trend': float(obs.tremor_trend),
        'side_effect_rate': float(obs.side_effect_rate),
        'gamma_arv': float(obs.gamma_arv),
        'dbs_entrainment': float(obs.dbs_entrainment),
        'stim_washout': float(obs.stim_washout),
        'target_output': float(obs.target_output),
        'tracking_accuracy': float(obs.tracking_accuracy),
    }

def build_user_prompt(step: int, obs_d: dict, task_id: str, history: list[str]) -> str:
    recent = '\n'.join(history[-4:]) if history else '(first step)'
    return textwrap.dedent(f"""
        Task: {task_id}
        Step: {step}
        Context: {_TASK_CONTEXT.get(task_id, '')}

        Brain state:
          beta_arv:         {obs_d['beta_arv']:.4f}
          tremor_arv:       {obs_d['tremor_arv']:.4f}
          force_preserved:  {obs_d['force_preserved']:.4f}
          side_effect_load: {obs_d['side_effect_load']:.4f}
          beta_trend:       {obs_d['beta_trend']:+.4f}
          tremor_trend:     {obs_d['tremor_trend']:+.4f}
          side_effect_rate: {obs_d['side_effect_rate']:+.4f}
          gamma_arv:        {obs_d['gamma_arv']:.4f}
          dbs_entrainment:  {obs_d['dbs_entrainment']:.4f}
          stim_washout:     {obs_d['stim_washout']:.4f}
          target_output:    {obs_d['target_output']:.4f}
          tracking_accuracy:{obs_d['tracking_accuracy']:.4f}

        Hints:
        - worsening symptoms + acceptable safety => increase a little
        - high gamma or high side-effect burden => reduce
        - improving symptoms + rising burden => hold or taper
        - stable control => use the lowest effective dose

        Recent steps:
        {recent}

        Output JSON only now.
    """).strip()

def parse_action(text: str) -> Optional[dict]:
    text = (text or '').strip()
    s = text.find('{'); e = text.rfind('}') + 1
    if s == -1 or e == 0:
        return None
    try:
        return json.loads(text[s:e])
    except Exception:
        return None

def action_from_dict(d: Optional[dict], target_output: float = 0.0) -> ParkinsonsMotorAction:
    if not d:
        return ParkinsonsMotorAction(
            motor_command=float(max(-1.0, min(1.0, target_output))),
            dbs_amplitude=1.0, dbs_pulse_width=0.13, dbs_frequency=130.0,
        )
    return ParkinsonsMotorAction(
        motor_command=float(max(-1.0, min(1.0, target_output))),
        dbs_amplitude=float(max(0.0, min(5.0,  d.get('dbs_amplitude',   1.0)))),
        dbs_pulse_width=float(max(0.06, min(0.20, d.get('dbs_pulse_width', 0.13)))),
        dbs_frequency=float(max(60.0, min(185.0, d.get('dbs_frequency',  130.0)))),
    )

# Quick smoke test: env can reset+step
_env = ParkinsonsMotorEnvironment()
_obs = _env.reset(task_id='easy', seed=0)
print(f"[smoke] easy: episode_steps={_obs.metadata['episode_steps']} target_output={_obs.metadata['target_output']:.2f}")
_obs = _env.step(action_from_dict({'dbs_amplitude':1.0,'dbs_pulse_width':0.13,'dbs_frequency':130.0}, _obs.target_output))
print(f"[smoke] step ok: reward={_obs.reward:.3f} done={_obs.done}")

## 6 · Heuristic baseline (sanity check + SFT teacher)

We use the project's `safety_aware` policy as the teacher. Episodes that score above threshold become SFT data; the rest are dropped (rejection sampling).

In [ ]:
from parkinsons_Motor.evaluation.eval_suite import safety_aware_policy
import statistics

def run_policy_episode(policy_fn, task_id: str, seed: int, return_steps: bool = False):
    env = ParkinsonsMotorEnvironment()
    obs = env.reset(task_id=task_id, seed=seed)
    n_steps = obs.metadata['episode_steps']
    history, steps_log, total_r = [], [], 0.0
    for t in range(n_steps):
        action = policy_fn(obs)
        obs_d = obs_to_dict(obs)
        prompt_text = build_user_prompt(t, obs_d, task_id, history)
        action_json = {
            'dbs_amplitude':   round(float(action.dbs_amplitude),   3),
            'dbs_pulse_width': round(float(action.dbs_pulse_width), 3),
            'dbs_frequency':   round(float(action.dbs_frequency),   1),
        }
        next_obs = env.step(action)
        history.append(f"step={t} action={json.dumps(action_json)}")
        if return_steps:
            steps_log.append({'prompt': prompt_text, 'action_json': action_json, 'reward': float(next_obs.reward)})
        total_r += float(next_obs.reward)
        obs = next_obs
    grader = float(obs.grader_score)
    success = bool(obs.episode_success)
    return {'task_id': task_id, 'seed': seed, 'grader': grader, 'success': success,
            'total_reward': total_r, 'n_steps': n_steps, 'steps': steps_log if return_steps else None}

print('=== safety_aware baseline ===')
for tid in CFG.public_tasks:
    rs = [run_policy_episode(safety_aware_policy, tid, s) for s in CFG.eval_seeds]
    g = [r['grader'] for r in rs]
    print(f"  {tid:6s}: grader mean={statistics.mean(g):.3f}  std={statistics.pstdev(g):.3f}  successes={sum(r['success'] for r in rs)}/{len(rs)}")

## 7 · Build the SFT dataset (rejection sampling)

Each example is `(prompt, JSON action)`. We **only keep steps from high‑scoring episodes** so the LM learns clinically reasonable behavior, not noise.

In [ ]:
import random, json, time

random.seed(7)
all_examples, ep_stats = [], []
t0 = time.time()
for tid, n_eps in CFG.sft_episodes_per_task.items():
    seeds = random.sample(range(1000), n_eps)
    kept_eps, kept_steps = 0, 0
    for s in seeds:
        ep = run_policy_episode(safety_aware_policy, tid, s, return_steps=True)
        ep_stats.append((tid, s, ep['grader']))
        if ep['grader'] < CFG.sft_min_grader_for_any_keep:
            continue
        # full keep above the higher threshold; otherwise keep top half by reward
        if ep['grader'] >= CFG.sft_min_grader_for_full_keep:
            chosen = ep['steps']
        else:
            srt = sorted(ep['steps'], key=lambda s: s['reward'], reverse=True)
            chosen = srt[: max(1, len(srt) // 2)]
        for st in chosen:
            all_examples.append({
                'task_id': tid, 'seed': s, 'grader': ep['grader'],
                'system': SYSTEM_PROMPT,
                'user':   st['prompt'],
                'assistant': json.dumps(st['action_json']),
            })
        kept_eps += 1; kept_steps += len(chosen)
    print(f"  task={tid:6s}: episodes kept {kept_eps}/{n_eps}  steps added {kept_steps}")

random.shuffle(all_examples)
all_examples = all_examples[: CFG.sft_max_examples]
print(f"\nTotal SFT examples: {len(all_examples)} (cap {CFG.sft_max_examples})")
print(f"Per-task grader: " + ', '.join(
    f"{t}={statistics.mean([g for tt,_,g in ep_stats if tt==t]):.3f}" for t in CFG.public_tasks))

with open(CFG.sft_data_path, 'w', encoding='utf-8') as f:
    for ex in all_examples:
        f.write(json.dumps(ex) + '\n')
print(f"Wrote: {CFG.sft_data_path}  ({time.time()-t0:.1f}s total)")

## 8 · Load Qwen2.5‑3B‑Instruct (4‑bit) + LoRA via Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = CFG.base_model,
    max_seq_length  = CFG.max_seq_len,
    dtype           = None,    # auto: fp16 on T4, bf16 on A100/L4
    load_in_4bit    = True,    # 4-bit NF4 base weights via bitsandbytes (LoRA stays fp16/bf16)
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Confirm 4-bit at runtime
n4 = sum(1 for _, m in model.named_modules() if 'Linear4bit' in type(m).__name__)
param_bytes = sum(p.numel() * (0.5 if 'Linear4bit' in type(p).__class__.__name__ else p.element_size()) for p in model.parameters())
print(f'4-bit linear layers: {n4}   model dtype: {next(model.parameters()).dtype}   approx weight memory: {param_bytes/1e9:.2f} GB')

model = FastLanguageModel.get_peft_model(
    model,
    r              = CFG.lora_r,
    lora_alpha     = CFG.lora_alpha,
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout   = 0.0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 7,
    use_rslora     = False,
)
model.print_trainable_parameters()
print('Loaded base + LoRA (default adapter).')

## 9 · SFT training (TRL `SFTTrainer`)

We format each example with the Qwen chat template and only train on the **assistant** tokens (mask the prompt).

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling, TrainerCallback

raw = [json.loads(l) for l in open(CFG.sft_data_path, encoding='utf-8')]
print('Loaded', len(raw), 'examples')

def to_chat(ex):
    msgs = [
        {'role': 'system', 'content': ex['system']},
        {'role': 'user',   'content': ex['user']},
        {'role': 'assistant', 'content': ex['assistant']},
    ]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

ds = Dataset.from_list(raw).map(to_chat, remove_columns=Dataset.from_list(raw).column_names)
print('Sample text:\n', ds[0]['text'][:600], '\n...')

class JsonlTrainerCallback(TrainerCallback):
    """Stream every Trainer log row to our SFT JSONL log."""
    def on_log(self, args, state, control, logs=None, **kw):
        if not logs:
            return
        LOG.sft_steps.log(
            step=int(state.global_step), epoch=float(state.epoch or 0.0),
            **{k: float(v) for k, v in logs.items() if isinstance(v, (int, float))},
        )

# One checkpoint per epoch (kept inside the run dir for resumability)
sft_args = SFTConfig(
    output_dir              = str(pathlib.Path(CFG.run_dir) / 'checkpoints/sft_epochs'),
    per_device_train_batch_size = CFG.sft_batch_size,
    gradient_accumulation_steps = CFG.sft_grad_accum,
    num_train_epochs        = CFG.sft_epochs,
    learning_rate           = CFG.sft_lr,
    lr_scheduler_type       = 'cosine',
    warmup_ratio            = CFG.sft_warmup_ratio,
    optim                   = 'adamw_8bit',
    logging_steps           = 5,
    save_strategy           = 'epoch',
    save_total_limit        = CFG.sft_epochs + 1,
    bf16                    = torch.cuda.is_bf16_supported(),
    fp16                    = not torch.cuda.is_bf16_supported(),
    report_to               = 'none',
    dataset_text_field      = 'text',
    max_seq_length          = CFG.max_seq_len,
    packing                 = False,
)

# TRL renamed `tokenizer` -> `processing_class` in 0.13. Try both for compat.
try:
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=ds, args=sft_args,
                         callbacks=[JsonlTrainerCallback()])
except TypeError:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=sft_args,
                         callbacks=[JsonlTrainerCallback()])

t0 = time.time()
train_out = trainer.train()
sft_wall_s = time.time() - t0
print(f'SFT done in {sft_wall_s/60:.1f} min. Final loss: {train_out.training_loss:.4f}')

# Persist final SFT adapter as the canonical "sft_final" checkpoint
model.save_pretrained(CFG.sft_adapter_dir)
tokenizer.save_pretrained(CFG.sft_adapter_dir)

# Persist the SFT training log to a stable location for plotting later
sft_log = [r for r in trainer.state.log_history if 'loss' in r]
with open(pathlib.Path(CFG.sft_adapter_dir) / 'sft_log_history.json', 'w') as f:
    json.dump(sft_log, f, indent=2)
with open(pathlib.Path(CFG.run_dir) / 'logs/sft/sft_summary.json', 'w') as f:
    json.dump({
        'final_loss': float(train_out.training_loss),
        'epochs': CFG.sft_epochs,
        'examples': len(raw),
        'wall_seconds': sft_wall_s,
        'steps': int(trainer.state.global_step),
    }, f, indent=2)
print(f'Saved SFT adapter -> {CFG.sft_adapter_dir}  (log entries: {len(sft_log)})')
print(f'Per-epoch checkpoints under {sft_args.output_dir}')

# Free SFT optimizer/grad memory before GRPO starts on the same GPU.
import gc
del trainer, train_out
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'GPU mem after SFT cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB used')


## 10 · Quick SFT eval (1 seed/task)

Sanity check that SFT didn't break the model. Also defines `model_policy()` reused later for GRPO rollouts and final eval.

In [ ]:
import torch
FastLanguageModel.for_inference(model)  # 2x faster generation

@torch.inference_mode()
def generate_action(model, tokenizer, system: str, user: str, max_new_tokens: int = 64,
                    temperature: float = 0.0, top_p: float = 1.0) -> str:
    msgs = [{'role':'system','content':system},{'role':'user','content':user}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 1e-5),
        top_p=top_p,
        pad_token_id=tokenizer.pad_token_id,
    )
    gen = out[0, inputs.input_ids.shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True)

def make_model_policy(model, tokenizer, temperature: float = 0.0):
    def _policy(obs):
        # closure stores per-episode history on the function attribute
        if not hasattr(_policy, '_history') or _policy._reset:
            _policy._history, _policy._step, _policy._reset = [], 0, False
        obs_d = obs_to_dict(obs)
        text = generate_action(
            model, tokenizer, SYSTEM_PROMPT,
            build_user_prompt(_policy._step, obs_d, obs.task_id, _policy._history),
            max_new_tokens=CFG.grpo_max_new_tok, temperature=temperature,
        )
        d = parse_action(text)
        action = action_from_dict(d, target_output=obs_d['target_output'])
        _policy._history.append(f"step={_policy._step} action={text.strip()[:120]}")
        _policy._step += 1
        return action
    _policy._reset = True
    def _reset_state():
        _policy._reset = True
    _policy.reset_state = _reset_state
    return _policy

def eval_model(model, tokenizer, tasks, seeds, temperature=0.0, label='model'):
    policy = make_model_policy(model, tokenizer, temperature=temperature)
    rows = []
    for tid in tasks:
        for s in seeds:
            policy.reset_state()
            r = run_policy_episode(policy, tid, s)
            rows.append(r)
            print(f"  [{label:>10s}] {tid:6s} seed={s}: grader={r['grader']:.3f} success={r['success']}")
    return rows

print('=== SFT model: greedy, 1 seed/task ===')
_sft_quick = eval_model(model, tokenizer, CFG.public_tasks, [0], temperature=0.0, label='SFT')

## 11 · GRPO training

Recipe (see notes in the planning summary):
- **Per‑step return‑to‑go advantages**, normalized within each (task, group) so credit lands on the actual decision step
- **Group of K trajectories** per update, all on the same task with different seeds
- **KL anchor to the SFT model** (loaded as a frozen second adapter `sft_ref`) to prevent the model from drifting away from JSON
- **Format reward**: small bonus if the response parses to valid JSON, small penalty otherwise — stops the model from collapsing to prose early in training
- **Curriculum**: easy → medium → hard, with optional early promotion when a task's rolling mean grader ≥ `grpo_promote_score`

In [ ]:
import torch, torch.nn.functional as F
import numpy as np

# Re-enable training mode on the model (was put in inference mode for the smoke eval)
FastLanguageModel.for_training(model)

# Load a SECOND copy of the SFT adapter as the frozen reference for KL
REF_ADAPTER_NAME = 'sft_ref'
if REF_ADAPTER_NAME not in getattr(model, 'peft_config', {}):
    model.load_adapter(CFG.sft_adapter_dir, adapter_name=REF_ADAPTER_NAME)
    print(f'Loaded reference adapter: {REF_ADAPTER_NAME}')
model.set_adapter('default')

# tokenizer needs a pad token + a sensible padding side for batched generation
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


# ============================================================================
# BATCHED HELPERS  (the whole point of this rewrite)
# ============================================================================

@torch.no_grad()
def sample_responses_batch(prompt_texts, temperature, max_new_tokens):
    """Batched generation: returns list of (prompt_ids, gen_ids, text) tuples.
    Uses LEFT-padding so the generation continues from the actual end of each prompt.
    """
    prev_side = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    enc = tokenizer(prompt_texts, return_tensors='pt', padding=True,
                    truncation=True, max_length=CFG.max_seq_len).to(model.device)
    tokenizer.padding_side = prev_side
    out = model.generate(
        input_ids=enc.input_ids, attention_mask=enc.attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 1e-3),
        top_p=0.95,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    L_prompt = enc.input_ids.shape[1]
    eos_id   = tokenizer.eos_token_id
    pad_id   = tokenizer.pad_token_id
    results = []
    for b in range(len(prompt_texts)):
        # strip left-padding from prompt
        attn = enc.attention_mask[b]
        first_real = int((attn == 1).nonzero()[0].item())
        prompt_actual = enc.input_ids[b, first_real:].detach().cpu()
        # generated tokens come after the (padded) prompt
        gen = out[b, L_prompt:].detach().cpu()
        # trim trailing padding / cut after first eos
        if eos_id is not None:
            eos_pos = (gen == eos_id).nonzero()
            if eos_pos.numel() > 0:
                gen = gen[:int(eos_pos[0].item()) + 1]
        if pad_id is not None and pad_id != eos_id:
            non_pad = (gen != pad_id).nonzero()
            if non_pad.numel() > 0:
                gen = gen[:int(non_pad[-1].item()) + 1]
            else:
                gen = gen[:1]
        text = tokenizer.decode(gen, skip_special_tokens=True)
        results.append((prompt_actual, gen, text))
    return results


def batch_per_token_logp(model, prompts, gens, *, train: bool):
    """Right-pad a batch of (prompt, gen) pairs and forward in one shot.
    Returns list of per-item per-token logp tensors (length = len(gen) for each item).
    """
    device = model.device
    pad_id = tokenizer.pad_token_id
    plens = [int(p.shape[0]) for p in prompts]
    glens = [int(g.shape[0]) for g in gens]
    seqs  = [torch.cat([p, g]) for p, g in zip(prompts, gens)]
    L = max(int(s.shape[0]) for s in seqs)
    B = len(seqs)
    input_ids = torch.full((B, L), pad_id, dtype=torch.long, device=device)
    attn      = torch.zeros((B, L), dtype=torch.long, device=device)
    for b, s in enumerate(seqs):
        sl = int(s.shape[0])
        input_ids[b, :sl] = s.to(device)
        attn[b, :sl] = 1
    if train:
        logits = model(input_ids=input_ids, attention_mask=attn).logits
    else:
        with torch.no_grad():
            logits = model(input_ids=input_ids, attention_mask=attn).logits
    out = []
    for b in range(B):
        plen, glen = plens[b], glens[b]
        slc = logits[b, plen-1:plen-1+glen, :]
        lp  = F.log_softmax(slc.float(), dim=-1)
        gen = gens[b].to(device)
        out.append(lp.gather(-1, gen.unsqueeze(-1)).squeeze(-1))
    return out


def rollout_episodes_group(task_id, seeds, temperature,
                            stage_idx=-1, update_idx=-1, log_first=False):
    """Run len(seeds) episodes in parallel by batching the per-step LLM call.
    Returns (group_steps, graders, summaries).
    """
    G = len(seeds)
    envs = [ParkinsonsMotorEnvironment() for _ in seeds]
    obss = [e.reset(task_id=task_id, seed=s) for e, s in zip(envs, seeds)]
    n_steps    = obss[0].metadata['episode_steps']
    histories  = [[] for _ in range(G)]
    steps_all  = [[] for _ in range(G)]
    env_r_sum  = [0.0] * G
    fmt_b_sum  = [0.0] * G
    n_parsed   = [0] * G
    t_ep0 = time.time()
    gen_bsz = CFG.grpo_gen_batch_size or G

    for t in range(n_steps):
        # Build prompts for all G rollouts for this timestep
        prompts = []
        for i, obs in enumerate(obss):
            obs_d = obs_to_dict(obs)
            msgs = [{'role':'system','content':SYSTEM_PROMPT},
                    {'role':'user','content':build_user_prompt(t, obs_d, task_id, histories[i])}]
            prompts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

        # Batched generation (split into chunks if gen_bsz < G)
        results = []
        for i0 in range(0, G, gen_bsz):
            results.extend(sample_responses_batch(
                prompts[i0:i0 + gen_bsz], temperature, CFG.grpo_max_new_tok))

        for i in range(G):
            prompt_ids, gen_ids, text = results[i]
            d = parse_action(text)
            obs_d_i = obs_to_dict(obss[i])
            action  = action_from_dict(d, target_output=obs_d_i['target_output'])
            next_obs = envs[i].step(action)
            env_r = float(next_obs.reward)
            fmt_b = CFG.grpo_format_bonus if d is not None else -CFG.grpo_format_penalty
            reward = env_r + fmt_b
            env_r_sum[i] += env_r; fmt_b_sum[i] += fmt_b; n_parsed[i] += int(d is not None)
            steps_all[i].append({
                'prompt_ids': prompt_ids, 'gen_ids': gen_ids, 'reward': reward,
                'parsed': d is not None,
            })
            if log_first and i == 0:
                LOG.grpo_steps.log(
                    stage=stage_idx, update=update_idx, group=0, task=task_id, seed=seeds[i],
                    step=t, parsed=bool(d is not None), env_reward=env_r, format_bonus=fmt_b,
                    action=d if d is not None else {},
                    obs={k: float(getattr(next_obs, k, 0.0)) for k in
                         ('beta_arv','tremor_arv','side_effect_load','force_preserved','gamma_arv')},
                    response_preview=text.strip()[:200],
                )
            histories[i].append(f"step={t} action={text.strip()[:120]}")
            obss[i] = next_obs

    graders, summaries = [], []
    wall = time.time() - t_ep0
    for i in range(G):
        grader = float(obss[i].grader_score) if obss[i].grader_score >= 0 else 0.0
        graders.append(grader)
        summ = {
            'task': task_id, 'seed': seeds[i], 'temperature': temperature,
            'stage': stage_idx, 'update': update_idx, 'group': i,
            'n_steps': n_steps, 'n_parsed': n_parsed[i],
            'parse_rate': n_parsed[i] / max(n_steps, 1),
            'env_reward_sum': env_r_sum[i], 'format_bonus_sum': fmt_b_sum[i],
            'reward_sum': env_r_sum[i] + fmt_b_sum[i],
            'grader': grader, 'success': bool(obss[i].episode_success),
            'wall_s': wall / G,  # per-rollout share of wall time
        }
        LOG.grpo_rollouts.log(**summ)
        summaries.append(summ)
    return steps_all, graders, summaries


def compute_advantages(group_steps_list, gamma: float):
    """Per-step return-to-go advantages, normalized within the group."""
    flat_advs = []
    for steps in group_steps_list:
        rewards = [s['reward'] for s in steps]
        rtg, running = [], 0.0
        for r in reversed(rewards):
            running = r + gamma * running
            rtg.append(running)
        rtg.reverse()
        flat_advs.extend(rtg)
    arr = np.array(flat_advs, dtype=np.float32)
    mu, sd = arr.mean(), arr.std() + 1e-6
    return ((arr - mu) / sd).tolist()


# Optimizer over LoRA params only
policy_params = [p for n, p in model.named_parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(policy_params, lr=CFG.grpo_lr, betas=(0.9, 0.95), weight_decay=0.0)
print(f'Optimizer over {sum(p.numel() for p in policy_params):,} trainable params')
print(f'Speed knobs: gen_batch={CFG.grpo_gen_batch_size or "group_size"}  '
      f'ref_micro_bsz={CFG.grpo_ref_micro_bsz}  pol_micro_bsz={CFG.grpo_pol_micro_bsz}  '
      f'max_new_tok={CFG.grpo_max_new_tok}  lora_r={CFG.lora_r}')


In [ ]:
import random, time
from collections import deque

history_log = []  # in-memory mirror of LOG.grpo_updates for plotting later

def save_grpo_checkpoint(tag: str):
    """Save the trainable adapter under checkpoints/grpo_<tag>/."""
    out = pathlib.Path(CFG.run_dir) / 'checkpoints' / f'grpo_{tag}'
    out.mkdir(parents=True, exist_ok=True)
    model.set_adapter('default')
    model.save_pretrained(str(out))
    tokenizer.save_pretrained(str(out))
    print(f'  [ckpt] saved -> {out}')


def grpo_update(task_id: str, group_size: int, sample_temp: float,
                stage_idx: int, update_idx: int):
    """One full GRPO update step on `task_id`, fully batched."""
    # ---- 1. Rollouts (BATCHED across the group) ----
    model.set_adapter('default')
    FastLanguageModel.for_inference(model)
    seeds = [random.randint(0, 10_000) for _ in range(group_size)]
    t_roll = time.time()
    group_steps, graders, summaries = rollout_episodes_group(
        task_id, seeds, sample_temp,
        stage_idx=stage_idx, update_idx=update_idx,
        log_first=(update_idx == 0),
    )
    n_gen_total = sum(len(st) for st in group_steps)
    rollout_s_per_step = (time.time() - t_roll) / max(1, n_gen_total)
    advs = compute_advantages(group_steps, gamma=CFG.grpo_gamma_rtg)
    flat = []
    idx = 0
    for steps in group_steps:
        for st in steps:
            flat.append((st['prompt_ids'], st['gen_ids'], advs[idx]))
            idx += 1
    random.shuffle(flat)

    # ---- 2. Reference logprobs (BATCHED, no grad) ----
    t_ref = time.time()
    model.set_adapter(REF_ADAPTER_NAME)
    ref_logps = [None] * len(flat)
    B_REF = max(1, CFG.grpo_ref_micro_bsz)
    for i in range(0, len(flat), B_REF):
        chunk = flat[i:i + B_REF]
        pids = [c[0] for c in chunk]
        gids = [c[1] for c in chunk]
        lps = batch_per_token_logp(model, pids, gids, train=False)
        for j, lp in enumerate(lps):
            ref_logps[i + j] = lp.detach()
    ref_s = time.time() - t_ref

    # ---- 3. Policy update (BATCHED, with grad) ----
    t_pol = time.time()
    model.set_adapter('default')
    FastLanguageModel.for_training(model)
    optimizer.zero_grad()
    pg_running, kl_running, n = 0.0, 0.0, 0
    N = max(1, len(flat))
    B_POL = max(1, CFG.grpo_pol_micro_bsz)
    for i in range(0, len(flat), B_POL):
        chunk = flat[i:i + B_POL]
        pids = [c[0] for c in chunk]
        gids = [c[1] for c in chunk]
        advs_b = [c[2] for c in chunk]
        pol_logps = batch_per_token_logp(model, pids, gids, train=True)
        batch_loss = None
        for j, lp in enumerate(pol_logps):
            ref_lp   = ref_logps[i + j].to(lp.device)
            pg = -float(advs_b[j]) * lp.sum()
            log_ratio = ref_lp - lp
            kl = (torch.exp(log_ratio) - log_ratio - 1).mean()
            term = (pg + CFG.grpo_kl_coef * kl) / N
            batch_loss = term if batch_loss is None else batch_loss + term
            pg_running += float(pg.detach()); kl_running += float(kl.detach()); n += 1
        if batch_loss is not None:
            batch_loss.backward()
    torch.nn.utils.clip_grad_norm_(policy_params, CFG.grpo_grad_clip)
    optimizer.step()
    optimizer.zero_grad()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    pol_s = time.time() - t_pol

    # ---- 4. Build per-update record ----
    parse_rate = float(np.mean([s['parse_rate']      for s in summaries]))
    env_reward = float(np.mean([s['env_reward_sum']  for s in summaries]))
    fmt_bonus  = float(np.mean([s['format_bonus_sum'] for s in summaries]))
    info = {
        'stage': stage_idx, 'update': update_idx, 'task': task_id, 'group_size': group_size,
        'mean_grader':  float(np.mean(graders)),
        'std_grader':   float(np.std(graders)),
        'best_grader':  float(np.max(graders)),
        'worst_grader': float(np.min(graders)),
        'success_rate': float(np.mean([s['success'] for s in summaries])),
        'parse_rate':   parse_rate,
        'env_reward':   env_reward,
        'format_bonus': fmt_bonus,
        'pg_loss': pg_running / max(1, n),
        'kl': kl_running / max(1, n),
        'n_steps': n,
        'gen_s_per_step': rollout_s_per_step,
        'ref_s': ref_s, 'pol_s': pol_s,
    }
    LOG.grpo_updates.log(**info)
    return info


# --------------------------- run the curriculum ----------------------------
rolling = {tid: deque(maxlen=5) for tid, *_ in CFG.grpo_curriculum}
t_start = time.time()
for stage_idx, (task_id, n_updates, group_size, sample_temp) in enumerate(CFG.grpo_curriculum):
    print(f"\n=== Stage {stage_idx+1}/{len(CFG.grpo_curriculum)}: task={task_id}  updates<={n_updates}  "
          f"group={group_size}  T={sample_temp} ===")
    for u in range(n_updates):
        info = grpo_update(task_id, group_size, sample_temp, stage_idx, u)
        rolling[task_id].append(info['mean_grader'])
        history_log.append({**info, 'wall_s': time.time() - t_start})
        print(f"  upd {u+1:02d}/{n_updates}: grader={info['mean_grader']:.3f}+-{info['std_grader']:.3f}  "
              f"best={info['best_grader']:.3f}  succ={info['success_rate']:.0%}  "
              f"parse={info['parse_rate']:.2f}  env_r={info['env_reward']:+.2f}  "
              f"pg={info['pg_loss']:+.3f}  kl={info['kl']:.4f}  | "
              f"gen={info['gen_s_per_step']:.2f}s/step  ref={info['ref_s']:.1f}s  pol={info['pol_s']:.1f}s")
        if (len(rolling[task_id]) == rolling[task_id].maxlen and
                float(np.mean(rolling[task_id])) >= CFG.grpo_promote_score):
            print(f"  -> rolling mean {np.mean(rolling[task_id]):.3f} >= {CFG.grpo_promote_score}; promoting.")
            break
    save_grpo_checkpoint(f'after_{task_id}')

print(f"\nGRPO done in {(time.time()-t_start)/60:.1f} min over {len(history_log)} updates.")
save_grpo_checkpoint('final')

import shutil
final_src = pathlib.Path(CFG.run_dir) / 'checkpoints/grpo_final'
if str(final_src) != CFG.grpo_adapter_dir:
    if pathlib.Path(CFG.grpo_adapter_dir).exists():
        shutil.rmtree(CFG.grpo_adapter_dir)
    shutil.copytree(final_src, CFG.grpo_adapter_dir)
print('Final GRPO adapter:', CFG.grpo_adapter_dir)
print('Per-stage checkpoints:', list((pathlib.Path(CFG.run_dir) / 'checkpoints').glob('grpo_*')))


## 12 · Training curves (SFT loss + GRPO progress)

Two figures, saved to `CFG.plots_dir` for embedding in the README:

- `sft_loss.png` — supervised fine-tuning loss vs steps
- `grpo_curves.png` — per-group mean grader, JSON parse rate, KL(policy ‖ ref),
   and policy-gradient loss across GRPO updates, broken out by curriculum stage.

In [ ]:
import json, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- SFT loss curve ----------------------------------------------------------
sft_log_path = pathlib.Path(CFG.sft_adapter_dir) / 'sft_log_history.json'
if sft_log_path.exists():
    sft_log = json.loads(sft_log_path.read_text())
    sft_steps = [r.get('step', i+1) for i, r in enumerate(sft_log)]
    sft_loss  = [r['loss'] for r in sft_log]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(sft_steps, sft_loss, marker='.', linewidth=1.5)
    ax.set_xlabel('SFT optimizer step')
    ax.set_ylabel('Cross-entropy loss')
    ax.set_title(f'SFT training loss - {CFG.base_model.split("/")[-1]}')
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f'{CFG.plots_dir}/sft_loss.png', dpi=140); plt.show()
else:
    print('No SFT log found at', sft_log_path)

# --- GRPO curves -------------------------------------------------------------
df = pd.DataFrame(history_log)
df['global_step'] = range(1, len(df) + 1)
df.to_csv(f'{CFG.plots_dir}/grpo_training_log.csv', index=False)

# Stage boundary markers (where the curriculum task changes)
stage_changes = []
for i in range(1, len(df)):
    if df.iloc[i]['task'] != df.iloc[i-1]['task']:
        stage_changes.append(i + 0.5)

def add_stage_lines(ax):
    for x in stage_changes:
        ax.axvline(x, color='black', linestyle=':', alpha=0.4, linewidth=1)
    # task labels at top of axis
    last = 0
    for i, x in enumerate(stage_changes + [len(df) + 0.5]):
        tid = df.iloc[max(0, int(x)-1)]['task']
        ax.text((last + x) / 2, ax.get_ylim()[1] * 0.97, str(tid),
                ha='center', va='top', fontsize=8, alpha=0.6)
        last = x

# --- Reward curve (the headline plot the user asked for) ---------------------
fig, ax = plt.subplots(figsize=(10, 5))
xs = df['global_step'].values
ax.plot(xs, df['mean_grader'].values, marker='o', linewidth=1.6, color='#2a7fd6', label='mean grader (group)')
ax.fill_between(xs,
                (df['mean_grader'] - df['std_grader']).clip(0, 1),
                (df['mean_grader'] + df['std_grader']).clip(0, 1),
                color='#2a7fd6', alpha=0.18, label='+-1 std across group')
ax.plot(xs, df['best_grader'].values,  linestyle='--', color='#1d5fa0', alpha=0.7, label='best in group')
ax.plot(xs, df['worst_grader'].values, linestyle=':',  color='#1d5fa0', alpha=0.5, label='worst in group')
ax.set_ylim(0, 1.0); ax.set_xlim(0.5, len(df) + 0.5)
add_stage_lines(ax)
ax.set_xlabel('GRPO update'); ax.set_ylabel('Grader score (0-1)')
ax.set_title('GRPO reward curve - mean grader per group, with curriculum stages marked')
ax.grid(alpha=0.3); ax.legend(loc='lower right', fontsize=9)
plt.tight_layout(); plt.savefig(f'{CFG.plots_dir}/grpo_reward_curve.png', dpi=140); plt.show()

# --- 4-panel diagnostics -----------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
panels = [
    ('env_reward',  'Env reward sum per episode (group mean)', 'GRPO update', 'Sum of env rewards'),
    ('parse_rate',  'JSON parse rate (group mean)',             'GRPO update', 'Fraction of valid actions'),
    ('kl',          'KL( policy || sft_ref )',                  'GRPO update', 'KL (k3 estimator)'),
    ('success_rate','Success rate (group mean)',                'GRPO update', 'Success fraction'),
]
for ax, (col, title, xl, yl) in zip(axes.flatten(), panels):
    for tid in df['task'].unique():
        sub = df[df['task'] == tid]
        ax.plot(sub['global_step'], sub[col], marker='o', label=tid)
    ax.set_title(title); ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.legend(title='Curriculum stage', fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{CFG.plots_dir}/grpo_curves.png', dpi=140); plt.show()

print('Saved:')
print('  -', f'{CFG.plots_dir}/sft_loss.png')
print('  -', f'{CFG.plots_dir}/grpo_reward_curve.png')
print('  -', f'{CFG.plots_dir}/grpo_curves.png')
print('  -', f'{CFG.plots_dir}/grpo_training_log.csv')


## 13 · Validation & Evaluation

Apples-to-apples comparison across **6 policies** (incl. random baseline) on the same task × seed grid:

| Policy             | What it is                                                        |
|--------------------|-------------------------------------------------------------------|
| `no_dbs`           | Constant zero stimulation. Worst-case clinical reference.         |
| `safety_aware`     | Hand-tuned heuristic from `evaluation/eval_suite.py`.             |
| `base_qwen`        | Vanilla `Qwen2.5-3B-Instruct`, no SFT, no GRPO. (LoRA disabled.)  |
| `sft_only`         | After SFT only — `sft_ref` adapter active.                        |
| `sft_plus_grpo`    | After SFT then GRPO — `default` adapter active.                   |

We collect:
1. **Final overall grader** + std across seeds, **success rate**
2. **Per-component scores** (force / beta / tremor / safety / smoothness / efficiency / terminal_stability / recovery)
3. **Per-step trajectories** for one demo seed per task — what the agent actually does to the patient

Plots saved (all labeled, ready for the README): `eval_grader_by_task.png`,
`eval_success_rate.png`, `eval_component_heatmap.png`, `eval_trajectories_<task>.png`,
`headline_before_after.png`.

In [ ]:
import torch, json, statistics, time
import numpy as np

# Eval-time signals we'll trace per step (used by both LLM and heuristic policies).
# Anything pulled off `obs` directly: brain biomarkers, force, side-effect load,
# tracking, plus actually-delivered DBS amplitude / pulse width / entrainment.
TRACE_KEYS = [
    'beta_arv', 'tremor_arv', 'force_preserved', 'side_effect_load',
    'gamma_arv', 'tracking_accuracy', 'dbs_amplitude_ma',
    'dbs_pulse_width_ms', 'dbs_entrainment',
]

# The actual evaluation primitives -- evaluate_heuristic + evaluate_llm_batched --
# live in the next cell, alongside the eval driver. They share `model`, `tokenizer`,
# `sample_responses_batch`, and `FastLanguageModel` from earlier cells.
print('Eval constants ready.  Will score 6 policies on',
      len(CFG.public_tasks), 'tasks x', len(CFG.eval_seeds), 'seeds =',
      len(CFG.public_tasks) * len(CFG.eval_seeds), 'episodes per LLM policy.')


In [ ]:
import time, statistics, json, pathlib
TRACE_SEED = CFG.eval_seeds[0]   # one seed per task gets full trajectory + inference log

eval_rows, eval_traces = [], {}

def make_random_policy():
    import random as _rng
    def _p(obs):
        return ParkinsonsMotorAction(
            motor_command=float(obs.target_output),
            dbs_amplitude=_rng.uniform(0.0, 2.5),
            dbs_pulse_width=_rng.uniform(0.06, 0.20),
            dbs_frequency=_rng.uniform(80.0, 170.0),
        )
    return _p

policies = [
    ('no_dbs',        'heuristic', lambda: (lambda obs: ParkinsonsMotorAction(
                          motor_command=float(obs.target_output),
                          dbs_amplitude=0.0, dbs_pulse_width=0.06, dbs_frequency=130.0))),
    ('random',        'heuristic', lambda: make_random_policy()),
    ('safety_aware',  'heuristic', lambda: safety_aware_policy),
    ('base_qwen',     'llm',       None),
    ('sft_only',      'llm',       None),
    ('sft_plus_grpo', 'llm',       None),
]


# ============================================================================
# Path A: existing per-episode loop for non-LLM policies (cheap, no GPU)
# ============================================================================
def evaluate_heuristic(policy_name, policy_fn, task_id, seed, capture_trace):
    env = ParkinsonsMotorEnvironment()
    obs = env.reset(task_id=task_id, seed=seed)
    n_steps = obs.metadata['episode_steps']
    if hasattr(policy_fn, 'reset_state'):
        policy_fn.reset_state()
    trace = {k: [] for k in TRACE_KEYS} if capture_trace else None
    if capture_trace:
        trace['reward'] = []
    for _ in range(n_steps):
        if capture_trace:
            for k in TRACE_KEYS:
                trace[k].append(float(getattr(obs, k, 0.0)))
        action = policy_fn(obs)
        obs = env.step(action)
        if capture_trace:
            trace['reward'].append(float(obs.reward))
    comp = obs.metadata.get('score_details', {}) or {}
    row = {
        'policy': policy_name, 'task': task_id, 'seed': seed,
        'grader': float(obs.grader_score),
        'success': bool(obs.episode_success),
        'n_steps': n_steps,
    }
    row.update({f'comp_{k}': float(v) for k, v in comp.items() if isinstance(v, (int, float))})
    return row, trace


# ============================================================================
# Path B: BATCHED LLM eval - runs every (task, seed) pair in lockstep, batching
# the per-step generate() call across them. This is the big speedup.
# ============================================================================
def evaluate_llm_batched(policy_name, variant, tasks, seeds, write_logs_for_seed):
    pairs = [(t, s) for t in tasks for s in seeds]
    G = len(pairs)
    if G == 0:
        return []

    # Set the right adapter ONCE for the whole batched run
    if   variant == 'base_qwen':     model.set_adapter('default'); disable_adapter = True
    elif variant == 'sft_only':      model.set_adapter('sft_ref'); disable_adapter = False
    else:                            model.set_adapter('default'); disable_adapter = False
    FastLanguageModel.for_inference(model)

    envs = [ParkinsonsMotorEnvironment() for _ in pairs]
    obss = [e.reset(task_id=t, seed=s) for e, (t, s) in zip(envs, pairs)]
    n_steps_per = [obs.metadata['episode_steps'] for obs in obss]
    histories = [[] for _ in range(G)]
    done      = [False] * G

    do_trace  = [s == write_logs_for_seed for (_, s) in pairs]
    traces    = [({k: [] for k in TRACE_KEYS} | {'reward': []}) if dt else None for dt in do_trace]
    write_log = [s == write_logs_for_seed for (_, s) in pairs]
    inf_buffers = [[] for _ in range(G)]

    eval_gen_bsz = CFG.grpo_gen_batch_size or G
    max_steps = max(n_steps_per)
    n_gen_calls = 0
    t0 = time.time()

    for t in range(max_steps):
        active = [i for i in range(G) if not done[i]]
        if not active:
            break

        # snapshot trace BEFORE step
        for i in active:
            if traces[i] is not None:
                for k in TRACE_KEYS:
                    traces[i][k].append(float(getattr(obss[i], k, 0.0)))

        # build prompts for every active episode
        prompts = []
        for i in active:
            obs_d = obs_to_dict(obss[i])
            user_p = build_user_prompt(t, obs_d, pairs[i][0], histories[i])
            msgs = [{'role':'system','content':SYSTEM_PROMPT},
                    {'role':'user','content':user_p}]
            prompts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

        # batched greedy generate (sub-batch by gen_bsz cap)
        results = []
        for i0 in range(0, len(prompts), eval_gen_bsz):
            chunk = prompts[i0:i0 + eval_gen_bsz]
            if disable_adapter and hasattr(model, 'disable_adapter'):
                with model.disable_adapter():
                    results.extend(sample_responses_batch(chunk, 0.0, CFG.grpo_max_new_tok))
            else:
                results.extend(sample_responses_batch(chunk, 0.0, CFG.grpo_max_new_tok))
            n_gen_calls += 1

        # apply each generation to its env
        for j, i in enumerate(active):
            prompt_ids, gen_ids, text = results[j]
            d = parse_action(text)
            obs_d_i = obs_to_dict(obss[i])
            action  = action_from_dict(d, target_output=obs_d_i['target_output'])
            next_obs = envs[i].step(action)

            if write_log[i]:
                inf_buffers[i].append({
                    'event': 'step', 'step': t,
                    'response_raw': text, 'parsed': d,
                    'action': {
                        'dbs_amplitude': float(action.dbs_amplitude),
                        'dbs_pulse_width': float(action.dbs_pulse_width),
                        'dbs_frequency':  float(action.dbs_frequency),
                        'motor_command':  float(action.motor_command),
                    },
                    'reward': float(next_obs.reward),
                    'done': bool(next_obs.done),
                    'obs': {k: float(getattr(next_obs, k, 0.0)) for k in
                            ('beta_arv','tremor_arv','side_effect_load','force_preserved',
                             'gamma_arv','tracking_accuracy')},
                })
            if traces[i] is not None:
                traces[i]['reward'].append(float(next_obs.reward))

            histories[i].append(f"step={t} action={text.strip()[:120]}")
            obss[i] = next_obs
            if t + 1 >= n_steps_per[i]:
                done[i] = True

    wall = time.time() - t0

    # Build rows + persist inference logs
    out = []
    for i in range(G):
        comp = obss[i].metadata.get('score_details', {}) or {}
        row = {
            'policy': policy_name, 'task': pairs[i][0], 'seed': pairs[i][1],
            'grader': float(obss[i].grader_score),
            'success': bool(obss[i].episode_success),
            'n_steps': n_steps_per[i],
        }
        row.update({f'comp_{k}': float(v) for k, v in comp.items() if isinstance(v, (int, float))})

        if write_log[i] and inf_buffers[i]:
            inf_path = (pathlib.Path(CFG.run_dir) / 'logs/eval' /
                        f'inference_{policy_name}_{pairs[i][0]}_seed{pairs[i][1]}.jsonl')
            with open(inf_path, 'w', encoding='utf-8') as f:
                f.write(json.dumps({
                    'event': 'start', 'policy': policy_name, 'task': pairs[i][0],
                    'seed': pairs[i][1], 'n_steps': n_steps_per[i],
                    'system_prompt': SYSTEM_PROMPT[:200] + '...',
                }) + '\n')
                for ev in inf_buffers[i]:
                    f.write(json.dumps(ev) + '\n')
                f.write(json.dumps({
                    'event': 'end', 'grader': row['grader'], 'success': row['success'],
                }) + '\n')
        out.append((row, traces[i]))

    print(f'    [batched] {policy_name}: {G} episodes, {n_gen_calls} batched generates, '
          f'{wall:.1f}s wall ({wall/G:.1f}s per episode)')
    return out


# ============================================================================
# Drive the eval
# ============================================================================
t0 = time.time()
for name, kind, build in policies:
    print(f'\n>>> Evaluating policy: {name}  ({kind})')
    if kind == 'llm':
        # batched path: ALL (task, seed) episodes for this policy in one go
        results = evaluate_llm_batched(name, name, CFG.public_tasks, CFG.eval_seeds,
                                        write_logs_for_seed=TRACE_SEED)
        for row, trace in results:
            eval_rows.append(row)
            LOG.eval_rows.log(**row)
            if trace is not None:
                eval_traces.setdefault(name, {})[row['task']] = trace
        for tid in CFG.public_tasks:
            sub = [r for r in eval_rows if r['policy'] == name and r['task'] == tid]
            gr  = [r['grader'] for r in sub]
            sr  = sum(r['success'] for r in sub) / max(len(sub), 1)
            print(f'    {tid:6s}: grader_mean={statistics.mean(gr):.3f}  '
                  f'std={statistics.pstdev(gr):.3f}  success={sr:.0%}')
    else:
        # heuristic path: per-episode, no GPU
        pf = build()
        for tid in CFG.public_tasks:
            for s in CFG.eval_seeds:
                capture = (s == TRACE_SEED)
                row, trace = evaluate_heuristic(name, pf, tid, s, capture_trace=capture)
                eval_rows.append(row)
                LOG.eval_rows.log(**row)
                if capture:
                    eval_traces.setdefault(name, {})[tid] = trace
            sub = [r for r in eval_rows if r['policy'] == name and r['task'] == tid]
            gr  = [r['grader'] for r in sub]
            sr  = sum(r['success'] for r in sub) / max(len(sub), 1)
            print(f'    {tid:6s}: grader_mean={statistics.mean(gr):.3f}  '
                  f'std={statistics.pstdev(gr):.3f}  success={sr:.0%}')
print(f'\nTotal eval wall-time: {(time.time()-t0)/60:.1f} min over {len(eval_rows)} episodes.')

# Build summary table (same shape as before so downstream cells keep working)
summary = {}
for name, _, _ in policies:
    summary[name] = {}
    for tid in CFG.public_tasks:
        sub = [r for r in eval_rows if r['policy'] == name and r['task'] == tid]
        gr  = [r['grader'] for r in sub]
        summary[name][tid] = {
            'mean_grader':  statistics.mean(gr) if gr else 0.0,
            'std_grader':   statistics.pstdev(gr) if gr else 0.0,
            'success_rate': sum(r['success'] for r in sub) / max(len(sub), 1),
            'n_episodes':   len(sub),
        }
    summary[name]['MEAN'] = {
        'mean_grader':  statistics.mean([summary[name][t]['mean_grader']  for t in CFG.public_tasks]),
        'success_rate': statistics.mean([summary[name][t]['success_rate'] for t in CFG.public_tasks]),
    }

print(f"\n{'policy':<16s}" + ''.join(f"{t:>14s}" for t in CFG.public_tasks) + f"{'MEAN':>14s}")
print('-' * (16 + 14 * (len(CFG.public_tasks) + 1)))
for name in summary:
    cells = [f"{summary[name][t]['mean_grader']:.3f} ({summary[name][t]['success_rate']:.0%})"
             for t in CFG.public_tasks]
    cells.append(f"{summary[name]['MEAN']['mean_grader']:.3f} "
                 f"({summary[name]['MEAN']['success_rate']:.0%})")
    print(f"{name:<16s}" + ''.join(f"{c:>14s}" for c in cells))

inf_files = sorted((pathlib.Path(CFG.run_dir) / 'logs/eval').glob('inference_*.jsonl'))
print(f'\nInference logs ({len(inf_files)}):')
for p in inf_files:
    print('  -', p.relative_to(CFG.run_dir))

# Drop stale activations / kv-cache before plotting + push.
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'GPU mem after eval cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB used')


### 13.1 · Headline before / after

One picture for the README hero: untrained base model vs the same model after SFT+GRPO.
If this bar gap is meaningful, the rest of the eval section explains *how* we got it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

before = [summary['base_qwen'][t]['mean_grader']     for t in CFG.public_tasks]
after  = [summary['sft_plus_grpo'][t]['mean_grader'] for t in CFG.public_tasks]
labels = CFG.public_tasks
x = np.arange(len(labels)); w = 0.38

fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - w/2, before, w, label='base Qwen-3B (before)',     color='#999999')
b2 = ax.bar(x + w/2, after,  w, label='SFT + GRPO (after)',         color='#2a7fd6')
for bars in (b1, b2):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Mean grader score (0-1)')
ax.set_ylim(0, 1.0)
ax.set_title(f'Before / after: {CFG.base_model.split("/")[-1]} on the Parkinson\'s Motor benchmark')
ax.grid(axis='y', alpha=0.3); ax.legend(loc='upper left')
plt.tight_layout(); plt.savefig(f'{CFG.plots_dir}/headline_before_after.png', dpi=140); plt.show()


### 13.2 · Per-task grader and success-rate bars (all 6 policies)

Same x-axis tasks, error bars = std across seeds. Shows where SFT alone is enough
and where GRPO actually buys you headroom. Success rate uses each task's own
`success_threshold`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

policy_order = ['no_dbs', 'safety_aware', 'base_qwen', 'sft_only', 'sft_plus_grpo']
colors = ['#bbbbbb', '#888888', '#d99c4f', '#4f9bd9', '#2a7fd6']
labels = CFG.public_tasks
x = np.arange(len(labels)); w = 0.8 / len(policy_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) grader
ax = axes[0]
for i, name in enumerate(policy_order):
    means = [summary[name][t]['mean_grader'] for t in labels]
    stds  = [summary[name][t]['std_grader']  for t in labels]
    ax.bar(x + i*w - 0.4 + w/2, means, w, yerr=stds, capsize=3, label=name, color=colors[i])
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Mean grader score')
ax.set_ylim(0, 1.0)
ax.set_title('Grader score by task and policy')
ax.grid(axis='y', alpha=0.3); ax.legend(fontsize=8)

# (b) success rate
ax = axes[1]
for i, name in enumerate(policy_order):
    sr = [summary[name][t]['success_rate'] for t in labels]
    ax.bar(x + i*w - 0.4 + w/2, sr, w, label=name, color=colors[i])
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Success rate (grader >= task threshold)')
ax.set_ylim(0, 1.05)
ax.set_title('Success rate by task and policy')
ax.grid(axis='y', alpha=0.3); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{CFG.plots_dir}/eval_grader_by_task.png', dpi=140)
plt.show()


### 13.3 · Per-component score breakdown

The grader is multi-objective: force preservation, beta suppression, tremor suppression,
tracking, safety budget, action smoothness, terminal stability, and recovery. This heatmap
shows where each policy gains/loses, averaged across tasks and seeds. Helps surface
*how* the trained model is winning (or where it's still weak).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df_eval = pd.DataFrame(eval_rows)
comp_cols = [c for c in df_eval.columns if c.startswith('comp_') and c not in
             ('comp_pre_penalty_score','comp_hard_failure_penalty','comp_overall_score',
              'comp_passes_safety_gate','comp_passes_symptom_gate','comp_passes_motor_gate',
              'comp_therapeutic_engagement')]
nice = {c: c.replace('comp_','').replace('_score','').replace('_',' ') for c in comp_cols}

policy_order = ['no_dbs', 'safety_aware', 'base_qwen', 'sft_only', 'sft_plus_grpo']
mat = []
for name in policy_order:
    sub = df_eval[df_eval['policy'] == name]
    mat.append([sub[c].mean() if c in sub else np.nan for c in comp_cols])
mat = np.array(mat)

fig, ax = plt.subplots(figsize=(1.4*len(comp_cols), 0.8*len(policy_order) + 1.2))
im = ax.imshow(mat, vmin=0, vmax=1, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(len(comp_cols))); ax.set_xticklabels([nice[c] for c in comp_cols], rotation=30, ha='right')
ax.set_yticks(range(len(policy_order))); ax.set_yticklabels(policy_order)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat[i, j]
        ax.text(j, i, f'{v:.2f}' if not np.isnan(v) else '-', ha='center', va='center',
                color='black' if 0.30 < v < 0.75 else 'white', fontsize=9)
ax.set_title('Per-component grader scores (mean across all tasks × seeds)')
fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02, label='Component score (0-1)')
plt.tight_layout()
plt.savefig(f'{CFG.plots_dir}/eval_component_heatmap.png', dpi=140)
plt.show()


### 13.4 · Before / after trajectories (what the agent *actually does*)

For one representative seed per task, plot the closed-loop signals over time:
neural symptoms (`beta_arv`, `tremor_arv`), safety budget (`side_effect_load` vs the
task's max), and the actual stimulation amplitude (`dbs_amplitude_ma`). This is the
qualitative "before vs after" story judges can read in seconds.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from parkinsons_Motor.tasks import get_task

policies_to_plot = ['safety_aware', 'base_qwen', 'sft_plus_grpo']
colors = {'safety_aware': '#888888', 'base_qwen': '#d99c4f', 'sft_plus_grpo': '#2a7fd6'}

for tid in CFG.public_tasks:
    if not all(tid in eval_traces.get(p, {}) for p in policies_to_plot):
        print(f'  no trace captured for {tid}; skipping'); continue
    task = get_task(tid)
    n = len(eval_traces[policies_to_plot[0]][tid]['beta_arv'])
    t_axis = np.arange(n)

    fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
    panels = [
        ('beta_arv',         'Beta-band oscillation (lower = better)', 0, 1.0, None),
        ('tremor_arv',       'Tremor envelope (lower = better)',         0, 1.0, None),
        ('side_effect_load', 'Side-effect load (vs task budget)',        0, 1.0, task.max_side_effect_load),
        ('dbs_amplitude_ma', 'Delivered DBS amplitude (mA)',             0, 5.0, getattr(task, 'amp_ceiling_ma', None)),
    ]
    for ax, (key, title, ymin, ymax, hline) in zip(axes.flatten(), panels):
        for p in policies_to_plot:
            tr = eval_traces[p][tid]
            ax.plot(t_axis, tr[key], label=p, color=colors[p], linewidth=1.6)
        if hline is not None:
            ax.axhline(hline, color='red', linestyle='--', linewidth=1, alpha=0.7,
                       label=f'budget={hline:.2f}' if 'budget' in title else f'cap={hline:.2f}')
        ax.set_title(title); ax.set_xlabel('Step (20 ms)'); ax.set_ylim(ymin, ymax)
        ax.grid(alpha=0.3); ax.legend(fontsize=8, loc='best')
    fig.suptitle(f'Closed-loop trajectory — task = {tid} (seed = {TRACE_SEED})', fontsize=12)
    plt.tight_layout()
    out = f'{CFG.plots_dir}/eval_trajectories_{tid}.png'
    plt.savefig(out, dpi=140); plt.show()
    print('  saved:', out)


### 13.5 · Save evaluation artifacts to disk

Everything ends up in `CFG.plots_dir`:
- `final_eval_summary.json` — table for the README / model card
- `final_eval_rows.csv` — every (policy, task, seed) row including all per-component scores
- `*.png` — labeled plots, ready to embed.

In [ ]:
import json, pandas as pd, pathlib

with open(CFG.eval_json, 'w') as f:
    json.dump(summary, f, indent=2)

pd.DataFrame(eval_rows).to_csv(f'{CFG.plots_dir}/final_eval_rows.csv', index=False)
with open(f'{CFG.plots_dir}/final_eval_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Wrote:')
print('  -', CFG.eval_json)
print('  -', f'{CFG.plots_dir}/final_eval_rows.csv')
print('  -', f'{CFG.plots_dir}/final_eval_summary.json')
print('PNG plots in', CFG.plots_dir + ':')
for p in sorted(pathlib.Path(CFG.plots_dir).glob('*.png')):
    print('  -', p.name)


### 13.6 · Logs & checkpoints index

Quick listing of everything this run produced — useful for the README and
for confirming all logs are on disk before pushing.

In [ ]:
import pathlib

run = pathlib.Path(CFG.run_dir)

def show_tree(label, root, max_depth=2):
    print(f'\n=== {label}: {root} ===')
    if not root.exists():
        print('  (missing)'); return
    for p in sorted(root.rglob('*')):
        rel = p.relative_to(root)
        depth = len(rel.parts) - 1
        if depth > max_depth: continue
        indent = '  ' * depth
        if p.is_dir():
            print(f'{indent}{rel.parts[-1]}/')
        else:
            sz = p.stat().st_size
            unit = 'B' if sz < 1024 else 'KB' if sz < 1024**2 else 'MB'
            sz_h = sz if unit == 'B' else (sz / 1024 if unit == 'KB' else sz / 1024**2)
            print(f'{indent}{rel.parts[-1]:<40s} {sz_h:>8.1f} {unit}')

show_tree('Logs',        run / 'logs',        max_depth=3)
show_tree('Checkpoints', run / 'checkpoints', max_depth=2)
show_tree('Plots',       run / 'plots',       max_depth=1)

# Also surface line counts of each JSONL log so we can spot empty files
print('\n=== JSONL row counts ===')
for p in sorted((run / 'logs').rglob('*.jsonl')):
    n = sum(1 for _ in open(p, encoding='utf-8'))
    print(f'  {p.relative_to(run)}: {n} rows')

# Close any open file handles on the loggers (frees buffers before HF push)
for _name, _l in vars(LOG).items():
    if hasattr(_l, 'close'):
        _l.close()
print('\nLoggers flushed and closed.')


### 13.7 · Training evidence summary (headline numbers)

Judges scan for **"X → Y improvement"** in 3 seconds. This cell prints the
headline deltas (base → SFT → GRPO) per task and saves them to
`logs/eval/submission_summary.json` so the README, blog, and video can quote
the exact same numbers.

In [ ]:
import json, pathlib, statistics

def get(name, tid, key='mean_grader'):
    return float(summary.get(name, {}).get(tid, {}).get(key, 0.0))

headline = {'per_task': {}, 'overall': {}}
print(f"\n{'task':<10s}{'random':>10s}{'no_dbs':>10s}{'safety':>10s}{'base':>10s}{'sft':>10s}{'sft+grpo':>12s}{'delta(base->grpo)':>22s}")
print('-' * 96)
for tid in CFG.public_tasks:
    rnd  = get('random',        tid)
    nod  = get('no_dbs',        tid)
    saf  = get('safety_aware',  tid)
    base = get('base_qwen',     tid)
    sft  = get('sft_only',      tid)
    fin  = get('sft_plus_grpo', tid)
    delta = fin - base
    pct = (delta / max(base, 1e-6)) * 100.0
    headline['per_task'][tid] = {
        'random': rnd, 'no_dbs': nod, 'safety_aware': saf,
        'base_qwen': base, 'sft_only': sft, 'sft_plus_grpo': fin,
        'delta_grader': delta, 'delta_pct': pct,
        'success_base':  get('base_qwen',     tid, 'success_rate'),
        'success_final': get('sft_plus_grpo', tid, 'success_rate'),
    }
    print(f"{tid:<10s}{rnd:>10.3f}{nod:>10.3f}{saf:>10.3f}{base:>10.3f}{sft:>10.3f}{fin:>12.3f}"
          f"{delta:>+10.3f}  ({pct:+5.1f}%)")

base_mean = statistics.mean([headline['per_task'][t]['base_qwen']     for t in CFG.public_tasks])
sft_mean  = statistics.mean([headline['per_task'][t]['sft_only']      for t in CFG.public_tasks])
fin_mean  = statistics.mean([headline['per_task'][t]['sft_plus_grpo'] for t in CFG.public_tasks])
succ_base = statistics.mean([headline['per_task'][t]['success_base']  for t in CFG.public_tasks])
succ_fin  = statistics.mean([headline['per_task'][t]['success_final'] for t in CFG.public_tasks])
headline['overall'] = {
    'mean_grader_base':  base_mean,
    'mean_grader_sft':   sft_mean,
    'mean_grader_final': fin_mean,
    'delta_grader_base_to_final': fin_mean - base_mean,
    'delta_pct_base_to_final':    (fin_mean - base_mean) / max(base_mean, 1e-6) * 100.0,
    'success_rate_base':  succ_base,
    'success_rate_final': succ_fin,
    'success_rate_delta': succ_fin - succ_base,
    'tasks_evaluated':    list(CFG.public_tasks),
    'seeds_per_task':     len(CFG.eval_seeds),
}

print(f"\n>>> HEADLINE: mean grader {base_mean:.3f} (base) -> {sft_mean:.3f} (SFT) -> {fin_mean:.3f} (SFT+GRPO)")
print(f">>> Absolute lift over base LLM: {fin_mean - base_mean:+.3f}  ({headline['overall']['delta_pct_base_to_final']:+.1f}%)")
print(f">>> Success rate: {succ_base:.0%} (base) -> {succ_fin:.0%} (SFT+GRPO)  ({(succ_fin - succ_base) * 100:+.0f} pts)")

out = pathlib.Path(CFG.run_dir) / 'logs/eval/submission_summary.json'
out.write_text(json.dumps(headline, indent=2))
print('\nSaved ->', out)


### 13.8 · Qualitative inference replay (what the agent actually does)

Numbers prove improvement; qualitative replay tells the *story*. This cell
loads the inference logs we wrote during evaluation and prints a side-by-side
walkthrough of `base_qwen` vs `sft_plus_grpo` for the same seed at three
checkpoints in the episode (start / mid / end). The output is also persisted
to `logs/eval/qualitative_replay.md` so it can be pasted directly into the
blog post or HF model card.

In [ ]:
import json, pathlib, textwrap

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]

def render_step(s, label):
    a = s['action']
    o = s['obs']
    return (f"  [{label}] step={s['step']:>3d}  "
            f"amp={a['dbs_amplitude']:.2f}mA  pw={a['dbs_pulse_width']*1000:.0f}us  "
            f"freq={a['dbs_frequency']:.0f}Hz  | "
            f"beta={o['beta_arv']:.3f}  tremor={o['tremor_arv']:.3f}  "
            f"side={o['side_effect_load']:.3f}  reward={s['reward']:+.3f}")

eval_dir = pathlib.Path(CFG.run_dir) / 'logs/eval'
TRACE_SEED = CFG.eval_seeds[0]
md_lines = ['# Qualitative inference replay\n',
            'Same patient, same seed, two policies. Look at how DBS amplitude and the',
            'beta/tremor biomarkers move differently after training.\n']

for tid in CFG.public_tasks:
    base_path  = eval_dir / f'inference_base_qwen_{tid}_seed{TRACE_SEED}.jsonl'
    final_path = eval_dir / f'inference_sft_plus_grpo_{tid}_seed{TRACE_SEED}.jsonl'
    if not base_path.exists() or not final_path.exists():
        print(f'(skipping {tid}: missing inference logs)'); continue
    base_log  = load_jsonl(base_path)
    final_log = load_jsonl(final_path)
    base_steps  = [r for r in base_log  if r.get('event') == 'step']
    final_steps = [r for r in final_log if r.get('event') == 'step']
    base_end  = next(r for r in base_log  if r.get('event') == 'end')
    final_end = next(r for r in final_log if r.get('event') == 'end')
    n = min(len(base_steps), len(final_steps))
    if n == 0: continue
    pick = [0, n // 2, n - 1]

    header = (f"\n=== Task: {tid}  (seed={TRACE_SEED}, {n} steps) ===\n"
              f"  base_qwen     final grader = {base_end['grader']:.3f}   success = {base_end['success']}\n"
              f"  sft_plus_grpo final grader = {final_end['grader']:.3f}   success = {final_end['success']}")
    print(header)
    md_lines.append(f'\n## Task: `{tid}` (seed {TRACE_SEED}, {n} steps)\n')
    md_lines.append(f"- **base_qwen**: grader = `{base_end['grader']:.3f}`, success = `{base_end['success']}`")
    md_lines.append(f"- **sft_plus_grpo**: grader = `{final_end['grader']:.3f}`, success = `{final_end['success']}`")
    md_lines.append('\n```')
    for i in pick:
        b = base_steps[i]; f = final_steps[i]
        line_b = render_step(b, 'BASE')
        line_f = render_step(f, 'FINAL')
        print(line_b); print(line_f); print()
        md_lines.append(line_b); md_lines.append(line_f); md_lines.append('')
    md_lines.append('```')

md_path = eval_dir / 'qualitative_replay.md'
md_path.write_text('\n'.join(md_lines), encoding='utf-8')
print('\nSaved markdown replay ->', md_path)


### 13.9 · Submission metadata (fill in your URLs)

Edit the cell below ONCE with your HF Space, blog, video, and slides URLs.
These get embedded into the HF model-card README and printed in the
submission checklist. Leave any of them as the default placeholder if you
don't have it yet — the checklist will flag them as missing.

In [ ]:
CFG.hf_env_space_url = 'https://huggingface.co/spaces/virustechhacks/parkinsons_Motor'  # the deployed env
CFG.hf_model_url     = f'https://huggingface.co/{CFG.hf_repo_id}'                       # set automatically from hf_repo_id
CFG.blog_url         = 'TODO: paste HF blog post URL here'
CFG.video_url        = 'TODO: paste YouTube/Loom video URL here'
CFG.slides_url       = 'TODO: paste Google Slides / pptx URL here'
CFG.team_name        = 'virustechhacks'
CFG.problem_theme    = 'Theme #3.1 - World Modeling (Professional Tasks)'

# Persist back to the run dir so the cfg.json snapshot reflects URLs
import json, pathlib
with open(pathlib.Path(CFG.run_dir) / 'cfg.json', 'w') as f:
    json.dump({k: v for k, v in vars(CFG).items() if not k.startswith('_')},
              f, indent=2, default=str)

print('Submission metadata:')
for k in ('team_name','problem_theme','hf_env_space_url','hf_model_url',
         'blog_url','video_url','slides_url'):
    print(f'  {k:<20s} = {getattr(CFG, k)}')


### 13.10 · Final submission checklist

Auto-runs against every judging-criteria minimum requirement and prints
what's done vs missing. Run this LAST, right before the HF push, so you
know exactly what's still needed.

In [ ]:
import json, pathlib

run = pathlib.Path(CFG.run_dir)
checks = []

def add(label, ok, note=''):
    checks.append({'ok': bool(ok), 'label': label, 'note': note})

# --- Minimum requirements ---
add('OpenEnv environment importable',
    True, 'parkinsons_Motor is importable in this notebook')
add('HF Space URL provided',
    not CFG.hf_env_space_url.startswith('TODO'),
    CFG.hf_env_space_url)
add('Training script (this Colab notebook)',
    True, 'dbs_sft_grpo_colab.ipynb')
add('SFT loss plot saved',
    (pathlib.Path(CFG.plots_dir) / 'sft_loss.png').exists(),
    'plots/sft_loss.png')
add('GRPO reward curve saved',
    (pathlib.Path(CFG.plots_dir) / 'grpo_reward_curve.png').exists(),
    'plots/grpo_reward_curve.png')
add('GRPO diagnostics plot saved',
    (pathlib.Path(CFG.plots_dir) / 'grpo_curves.png').exists(),
    'plots/grpo_curves.png')
add('Trained vs untrained baseline comparison',
    (pathlib.Path(CFG.plots_dir) / 'headline_before_after.png').exists(),
    'plots/headline_before_after.png')
add('Per-task bar comparison',
    (pathlib.Path(CFG.plots_dir) / 'eval_grader_by_task.png').exists(),
    'plots/eval_grader_by_task.png')
add('Per-component score heatmap',
    (pathlib.Path(CFG.plots_dir) / 'eval_component_heatmap.png').exists(),
    'plots/eval_component_heatmap.png')
add('Trajectory plots saved',
    any(pathlib.Path(CFG.plots_dir).glob('eval_trajectories_*.png')),
    'plots/eval_trajectories_*.png')
add('Submission summary JSON',
    (run / 'logs/eval/submission_summary.json').exists(),
    'logs/eval/submission_summary.json')
add('Qualitative replay markdown',
    (run / 'logs/eval/qualitative_replay.md').exists(),
    'logs/eval/qualitative_replay.md')
add('SFT adapter checkpoint',
    pathlib.Path(CFG.sft_adapter_dir).exists(),
    CFG.sft_adapter_dir)
add('GRPO per-stage checkpoints (>=1)',
    any((run / 'checkpoints').glob('grpo_after_*')),
    str(run / 'checkpoints'))
add('Final GRPO adapter',
    pathlib.Path(CFG.grpo_adapter_dir).exists(),
    CFG.grpo_adapter_dir)
add('Inference logs (LLM trace) for >= 1 task',
    any((run / 'logs/eval').glob('inference_sft_plus_grpo_*.jsonl')),
    'logs/eval/inference_*.jsonl')
add('Blog/HF post URL provided',
    not CFG.blog_url.startswith('TODO'),
    CFG.blog_url)
add('Video URL provided (<2 min)',
    not CFG.video_url.startswith('TODO'),
    CFG.video_url)
add('Slides URL provided',
    not CFG.slides_url.startswith('TODO'),
    CFG.slides_url)

# Random + 5 model policies all evaluated
expected_policies = {'random','no_dbs','safety_aware','base_qwen','sft_only','sft_plus_grpo'}
# (policies tuple is now (name, kind, build); just reference names by string set)
have_policies = set(summary.keys()) if 'summary' in dir() else set()
add('All 6 policies evaluated (incl. random baseline)',
    expected_policies.issubset(have_policies),
    f'have: {sorted(have_policies)}')

# Headline improvement is positive
try:
    head = json.loads((run / 'logs/eval/submission_summary.json').read_text())
    delta = head['overall']['delta_grader_base_to_final']
    add('GRPO improves over base LLM (delta > 0)', delta > 0,
        f'delta = {delta:+.3f} ({head["overall"]["delta_pct_base_to_final"]:+.1f}%)')
except Exception as e:
    add('GRPO improves over base LLM (delta > 0)', False, f'(could not compute: {e})')

print(f"\n{'='*72}")
print(f'  SUBMISSION CHECKLIST  ({sum(c["ok"] for c in checks)}/{len(checks)} passed)')
print(f"{'='*72}")
for c in checks:
    mark = '[x]' if c['ok'] else '[ ]'
    print(f'  {mark}  {c["label"]:<48s}  {c["note"]}')
print(f"{'='*72}")
missing = [c for c in checks if not c['ok']]
if missing:
    print(f'\n  STILL TO DO ({len(missing)}):')
    for c in missing:
        print(f'    - {c["label"]}')
else:
    print('\n  All submission requirements met. Ready to push.')

# Persist for the model card
(run / 'logs/eval/submission_checklist.json').write_text(
    json.dumps({'checks': checks, 'urls': {
        'hf_env_space': CFG.hf_env_space_url,
        'hf_model':     CFG.hf_model_url,
        'blog':         CFG.blog_url,
        'video':        CFG.video_url,
        'slides':       CFG.slides_url,
    }}, indent=2))


## 14 · Save merged weights and push to the Hugging Face Hub

Edit `CFG.hf_repo_id` above first (e.g. `your-username/dbs-qwen3b-sft-grpo`).

In [ ]:
from huggingface_hub import HfApi, create_repo
import json, pathlib

if not CFG.hf_repo_id or '/' not in CFG.hf_repo_id:
    raise RuntimeError('Set CFG.hf_repo_id to "<user>/<repo>" before pushing.')

api = HfApi()
create_repo(CFG.hf_repo_id, exist_ok=True, repo_type='model')

# Save merged 16-bit weights (so anyone can load with vanilla transformers)
print('Merging adapter into base weights (16-bit) ...')
model.save_pretrained_merged(CFG.merged_dir, tokenizer, save_method='merged_16bit')

print(f'Uploading merged weights to https://huggingface.co/{CFG.hf_repo_id} ...')
api.upload_folder(folder_path=CFG.merged_dir, repo_id=CFG.hf_repo_id,
                  repo_type='model', ignore_patterns=['*.bin'])

# Upload training + eval logs (small JSONL/JSON files)
for log_path in sorted(pathlib.Path(CFG.run_dir).rglob('logs/**/*.json*')):
    rel = log_path.relative_to(CFG.run_dir)
    api.upload_file(path_or_fileobj=str(log_path), path_in_repo=str(rel), repo_id=CFG.hf_repo_id)

# Upload run config snapshot
cfg_dump = pathlib.Path(CFG.run_dir) / 'cfg.json'
if cfg_dump.exists():
    api.upload_file(path_or_fileobj=str(cfg_dump), path_in_repo='cfg.json', repo_id=CFG.hf_repo_id)

# Upload all plots
for png in sorted(pathlib.Path(CFG.plots_dir).glob('*.png')):
    api.upload_file(path_or_fileobj=str(png), path_in_repo=f'plots/{png.name}', repo_id=CFG.hf_repo_id)
for csv in sorted(pathlib.Path(CFG.plots_dir).glob('*.csv')):
    api.upload_file(path_or_fileobj=str(csv), path_in_repo=f'plots/{csv.name}', repo_id=CFG.hf_repo_id)

# Compose enriched README ----------------------------------------------------
sub_path  = pathlib.Path(CFG.run_dir) / 'logs/eval/submission_summary.json'
repl_path = pathlib.Path(CFG.run_dir) / 'logs/eval/qualitative_replay.md'
sub  = json.loads(sub_path.read_text())  if sub_path.exists()  else {'overall': {}, 'per_task': {}}
repl = repl_path.read_text(encoding='utf-8') if repl_path.exists() else ''

ov = sub.get('overall', {})
headline_md = (
    f"**Mean grader (averaged over {len(CFG.public_tasks)} tasks x {len(CFG.eval_seeds)} seeds):** "
    f"`{ov.get('mean_grader_base', 0):.3f}` (base) -> "
    f"`{ov.get('mean_grader_sft', 0):.3f}` (SFT) -> "
    f"`{ov.get('mean_grader_final', 0):.3f}` (SFT+GRPO)  "
    f"**({ov.get('delta_grader_base_to_final', 0):+.3f}, {ov.get('delta_pct_base_to_final', 0):+.1f}%)**\n\n"
    f"**Success rate:** `{ov.get('success_rate_base', 0):.0%}` -> "
    f"`{ov.get('success_rate_final', 0):.0%}`  "
    f"(`{(ov.get('success_rate_delta', 0)) * 100:+.0f}` percentage points)"
)

# Per-task table for the README
rows = []
for tid in CFG.public_tasks:
    pt = sub.get('per_task', {}).get(tid, {})
    rows.append(f"| `{tid}` | {pt.get('random', 0):.3f} | {pt.get('no_dbs', 0):.3f} | "
                f"{pt.get('safety_aware', 0):.3f} | {pt.get('base_qwen', 0):.3f} | "
                f"{pt.get('sft_only', 0):.3f} | **{pt.get('sft_plus_grpo', 0):.3f}** | "
                f"**{pt.get('delta_grader', 0):+.3f}** |")
table_md = (
    "| Task | random | no_dbs | safety_aware | base_qwen | sft_only | sft+GRPO | delta(base->grpo) |\n"
    "|------|-------:|-------:|-------------:|----------:|---------:|---------:|------------------:|\n"
    + '\n'.join(rows)
)

base_model = CFG.base_model.split('/')[-1]
readme = f"""---
license: apache-2.0
base_model: {CFG.base_model}
tags:
- parkinsons
- deep-brain-stimulation
- reinforcement-learning
- grpo
- unsloth
- openenv
---

# DBS Agent: SFT -> GRPO on the Parkinson's Motor Environment

LLM agent ({base_model} + LoRA) trained as an adaptive Deep Brain Stimulation programmer
for the [Parkinson's Motor OpenEnv environment]({CFG.hf_env_space_url}).

## Headline result

{headline_md}

## Per-task scores (mean grader, higher is better)

{table_md}

## Submission links

- **Environment (HF Space):** {CFG.hf_env_space_url}
- **Trained model (this repo):** {CFG.hf_model_url}
- **Blog post:** {CFG.blog_url}
- **Video walkthrough:** {CFG.video_url}
- **Slide deck:** {CFG.slides_url}

## Training pipeline

1. **SFT warm-start** on rejection-sampled rollouts from the `safety_aware` heuristic policy
   ({CFG.sft_episodes_per_task} eps/task across {len(CFG.public_tasks)} tasks; kept fully when grader >= {CFG.sft_min_grader_for_full_keep}, top half kept when grader >= {CFG.sft_min_grader_for_any_keep}).
2. **GRPO** with per-step return-to-go advantages, group normalization, KL anchor
   to the frozen SFT reference adapter, and a small JSON-format bonus.
   Curriculum: easy -> medium -> hard, promoted by rolling mean grader.

## Plots

| Plot | Description |
|---|---|
| `plots/sft_loss.png` | Cross-entropy loss across SFT optimizer steps |
| `plots/grpo_reward_curve.png` | Mean grader per group with +-1 std band, stage boundaries marked |
| `plots/grpo_curves.png` | Env reward / parse rate / KL / success rate per stage |
| `plots/headline_before_after.png` | Base LLM vs SFT+GRPO mean grader |
| `plots/eval_grader_by_task.png` | Grader and success rate across all 6 policies and 3 tasks |
| `plots/eval_component_heatmap.png` | 8 grader components x 6 policies |
| `plots/eval_trajectories_*.png` | Per-step beta / tremor / side-effect / DBS amplitude |

## Logs (full reproducibility)

- `logs/sft/sft_steps.jsonl` - one row per SFT optimizer step
- `logs/grpo/rollouts.jsonl` - one row per GRPO rollout episode
- `logs/grpo/updates.jsonl` - one row per GRPO update step
- `logs/eval/episode_rows.jsonl` - one row per eval episode (all policies)
- `logs/eval/inference_*.jsonl` - full LLM trace (prompt -> response -> action -> obs) for replay
- `logs/eval/submission_summary.json` - the headline numbers above
- `logs/eval/submission_checklist.json` - submission-requirement audit

## Qualitative replay (base vs trained, same seed)

{repl[:4000] if repl else '_(qualitative replay not available)_'}

## How to load

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained('{CFG.hf_repo_id}')
mdl = AutoModelForCausalLM.from_pretrained('{CFG.hf_repo_id}', torch_dtype='auto', device_map='auto')
```

## Reproducing this run

The full Colab notebook is `dbs_sft_grpo_colab.ipynb` in the env repo.
The exact config used is checked into this model repo as `cfg.json`.
"""
(pathlib.Path(CFG.run_dir) / 'README_model_card.md').write_text(readme, encoding='utf-8')
api.upload_file(path_or_fileobj=readme.encode('utf-8'),
                path_in_repo='README.md', repo_id=CFG.hf_repo_id)
print(f'\nDone. View at: https://huggingface.co/{CFG.hf_repo_id}')


## 15 · (Optional) Reload from the Hub and re‑verify

Quick sanity check that the pushed weights work end‑to‑end.

In [ ]:
# Free the LoRA model before loading a SECOND full model from the Hub.
# Without this, T4 (16 GB) will OOM since we'd be holding both model + merged copy.
import gc, torch
try:
    del model
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from transformers import AutoTokenizer, AutoModelForCausalLM

print(f'Reloading {CFG.hf_repo_id} from the Hub for end-to-end verification ...')
tok2 = AutoTokenizer.from_pretrained(CFG.hf_repo_id)
m2   = AutoModelForCausalLM.from_pretrained(
    CFG.hf_repo_id,
    torch_dtype=torch.float16,
    device_map='auto',
    low_cpu_mem_usage=True,
)
m2.eval()

policy = make_model_policy(m2, tok2, temperature=0.0)
for tid in CFG.public_tasks:
    policy.reset_state()
    r = run_policy_episode(policy, tid, seed=42)
    print(f"  reload-check {tid:6s} seed=42: grader={r['grader']:.3f} success={r['success']}")
print('\nAll good - your DBS controller is live on the Hub.')
